# Pré-processamento

Clusterizar é agrupar municípios parecidos entre si, e "parecido" vira, na
prática, uma conta de distância entre duas listas de 22 números. O problema
é que essa conta só faz sentido se as 22 colunas estiverem em condições
comparáveis, e não estão: umas vão de 0 a 950, outras de 0 a 100 e outras
de 1 a 5. Somar tudo assim deixaria a coluna de maior escala decidir quase
sozinha quem se parece com quem.

Este notebook deixa a base nessas condições, em quatro passos, e gera a
**Figura 3** do artigo:

1. **`log1p`** nas variáveis de cauda longa, para os poucos municípios com
   valores altíssimos não dominarem a conta (decidido no notebook 02);
2. **padronização**, para todas as colunas passarem a falar a mesma língua;
3. **peso por bloco**, para as três fontes de dados pesarem igual;
4. **PCA**, que aqui não transforma nada, serve para diagnosticar quantas
   dimensões os dados realmente têm.

A saída é o `matriz_modelagem.csv`, o arquivo que a próxima entrega vai usar
para rodar a clusterização. Nenhum agrupamento acontece aqui.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSED
from estilo import aplicar_estilo, salvar
from figuras import plot_orcamento_blocos, plot_variancia_pca, plot_pc1_pc2

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)

## 1. Features usadas

A lista de features sai do dicionário, como no notebook 02, e a capital fica
de fora por não ter IEGM (a coluna `flag_sem_iegm` marca isso). Sobram 644
municípios e 22 features: 9 de criminalidade, 6 socioeconômicas e 7 de
gestão.

A lista `log_cols` separa as colunas que vão receber `log1p`: as 9 taxas
criminais e as 2 variáveis em reais, que são as de cauda longa identificadas
no notebook 02. Percentuais e notas de 1 a 5 não precisam, porque já vivem
numa faixa curta e sem valores extremos.

In [ ]:
bloco_de = dic.set_index("coluna")["bloco"]
ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]
# Mesma ordenação por bloco do notebook 02: mantém as colunas na mesma ordem
# em todas as tabelas e figuras do projeto.
features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],
                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))

# reset_index porque tirar a capital abre um buraco na numeração das linhas,
# e mais para baixo as posições precisam bater com as da matriz do sklearn.
modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)

# As duas listas dividem as colunas em "recebe log1p" e "não recebe".
log_cols = ([c for c in features if bloco_de[c] == "criminalidade"]
            + ["pib_percapita", "renda_domiciliar_mediana"])
demais_cols = [c for c in features if c not in log_cols]
print(f"{len(modelagem)} municípios | {len(features)} features | "
      f"log1p em {len(log_cols)}")

## 2. Transformar e padronizar

**Padronizar** uma coluna é subtrair a média dela e dividir pelo desvio
padrão. Depois disso a coluna fica com média 0 e desvio 1, e cada valor
passa a significar "tantos desvios acima ou abaixo da média" em vez de
reais, porcentagem ou ocorrências por 100 mil. É isso que torna 22 colunas
de unidades diferentes somáveis numa mesma distância..

O `ColumnTransformer` é o jeito do `scikit-learn` de aplicar tratamentos
diferentes a grupos de colunas diferentes numa passada só: aqui, `log1p`
seguido de padronização nas colunas de cauda longa, e só padronização nas
demais. Escrever assim, em vez de fazer as contas na mão, deixa o objeto
pronto para ser reaplicado do mesmo jeito na clusterização.

Um detalhe que importa: a média e o desvio de cada coluna são calculados
**nos 644 municípios que vão ser clusterizados**, e não na base inteira. Se
a capital entrasse nessa conta, ela mexeria na escala de todo mundo sem nem
participar do agrupamento.

In [ ]:
# Cada tupla do ColumnTransformer é (nome, o que fazer, em quais colunas).
pre = ColumnTransformer([
    # Cauda longa: primeiro log1p, depois padroniza (Pipeline = em sequência).
    ("log", Pipeline([("log1p", FunctionTransformer(np.log1p)),
                      ("z", StandardScaler())]), log_cols),
    # O resto (percentuais e notas de 1 a 5): só padroniza.
    ("z", StandardScaler(), demais_cols),
])
# fit_transform faz as duas coisas: aprende média e desvio de cada coluna e
# já aplica. O resultado sai na ordem em que os grupos foram declarados, por
# isso o [features] no fim recoloca tudo na ordem por bloco.
Z = pd.DataFrame(pre.fit_transform(modelagem[features]),
                 columns=log_cols + demais_cols)[features]
# Conferência: toda coluna tem que sair daqui com média 0 e desvio 1.
Z.describe().round(2).loc[["mean", "std"]]

## 3. Peso por bloco

Ficou um desequilíbrio que a padronização não resolve. Depois dela cada
**coluna** vale o mesmo, mas os blocos têm quantidades diferentes de
colunas: 9 de criminalidade, 6 socioeconômicas e 7 de gestão. Como a
distância soma a contribuição de todas as colunas, o bloco com mais colunas
puxa mais — e ele tem mais colunas por acaso, porque a fonte divulga os
dados assim, não porque criminalidade importe mais que gestão.

A correção é multiplicar cada coluna por `1/√n`, sendo `n` o número de
colunas do bloco dela. Funciona porque as variâncias se somam: um bloco de
`n` colunas padronizadas contribui com `n`, e dividir cada coluna por `√n`
faz a contribuição do bloco virar 1, seja ele de 6 ou de 9 colunas. Os três
passam a valer um terço cada.

A ideia vem da Análise Fatorial Múltipla (Escofier e Pagès, 1994), método
criado justamente para juntar grupos de variáveis de origens diferentes sem
deixar um grupo dominar. Vale registrar a diferença, porque não é a mesma
conta: a AFM divide cada grupo pela raiz do seu primeiro autovalor, o que
iguala o quanto cada grupo pode pesar **no primeiro eixo**; nós dividimos
por `√n`, o que iguala a **variância total** de cada bloco. A nossa versão é
mais simples de explicar e de conferir, e entrega o que queríamos: um terço
da distância para cada bloco, como mostra a figura abaixo.

In [ ]:
# Quantas colunas tem cada bloco (9, 6 e 7).
n_bloco = pd.Series([bloco_de[c] for c in features]).value_counts()
# Cada coluna recebe o peso do bloco a que pertence.
peso = pd.Series({c: 1 / np.sqrt(n_bloco[bloco_de[c]]) for c in features})
W = Z * peso        # W é a matriz final: padronizada e com peso aplicado
peso.groupby(bloco_de).first().round(3)

In [ ]:
def orcamento(M):
    """% da variância total que cada bloco tem."""
    # Com as colunas padronizadas, a variância de cada uma é o quanto ela
    # "ocupa" na distância. Somando por bloco dá para ver se os três estão
    # em pé de igualdade. ddof=0 usa a fórmula populacional, que é a mesma
    # que o StandardScaler usa.
    var = M.var(ddof=0)
    return (var.groupby(M.columns.map(bloco_de)).sum()
               .reindex(ORDEM_BLOCOS) / var.sum() * 100).round(1)

# Compara o antes (Z, só padronizada) com o depois (W, já com peso).
fig = plot_orcamento_blocos(orcamento(Z), orcamento(W))
salvar(fig, "figura_orcamento_blocos")

O gráfico mostra o antes e o depois. Sem peso, a criminalidade fica com
40,9% da distância, a gestão com 31,8% e o bloco socioeconômico com 27,3% —
uma diferença que vem só da quantidade de colunas de cada um. Com o peso, os
três ficam em 33,3%, exatamente o que a conta do `1/√n` promete.

## 4. Figura 3 - quantas dimensões os dados têm de verdade

O PCA (Análise de Componentes Principais) procura combinações das 22 colunas
que concentrem o máximo possível da variação. A primeira componente é a
direção em que os municípios mais se espalham; a segunda é a direção
seguinte que mais espalha, entre as que são independentes da primeira; e
assim por diante.

O número que interessa aqui é **quantas componentes são necessárias para
representar 80% da variação**. Se bastassem três ou quatro, seria sinal de
que as 22 colunas são, no fundo, poucas ideias repetidas de formas
diferentes. Se forem muitas, cada bloco está trazendo informação própria.

Neste notebook o PCA é só diagnóstico: a clusterização vai rodar sobre as 22
colunas com peso, não sobre as componentes.

In [ ]:
# PCA() sem n_components calcula todas as componentes, que é o que queremos
# para olhar a variância acumulada.
pca = PCA().fit(W)
fig = plot_variancia_pca(pca.explained_variance_ratio_)
salvar(fig, "figura3_variancia_pca")

São necessárias **12 das 22 componentes** para chegar a 80% da variação, e a
primeira delas sozinha explica menos de um quarto. Duas leituras:

1. **Os três blocos não dizem a mesma coisa.** É a mesma conclusão a que a
   matriz de Spearman do notebook 02 chegou, agora por outro caminho: não
   existe um punhado de dimensões escondidas que resuma a base.
2. **Um aviso para a próxima entrega.** Dados espalhados em muitas dimensões
   são um cenário ruim para métodos de agrupamento baseados em densidade,
   como o DBSCAN. Quando há muitas dimensões, os pontos ficam todos mais ou
   menos à mesma distância uns dos outros, e o método tende a classificar
   quase tudo como ruído. Fica registrado aqui, antes de rodar, para não
   parecer desculpa inventada depois.

## 5. O que as duas primeiras componentes significam

Cada componente é uma soma das 22 colunas, cada uma com um peso diferente.
Esses pesos se chamam **cargas**, e são eles que dizem o que a componente
representa: as colunas com carga alta, positiva ou negativa, são as que mais
definem aquela direção. Cargas de sinais opostos na mesma componente
significam que ela está **contrastando** os dois grupos de variáveis.

A tabela abaixo mostra só as features com carga de 0,25 para cima em alguma
das três primeiras componentes, para a leitura não se perder no meio de 22
linhas.

In [ ]:
# components_ traz uma linha por componente; o .T vira para uma linha por
# feature, que é como a tabela fica legível.
cargas = pd.DataFrame(pca.components_[:3].T, index=features,
                      columns=["PC1", "PC2", "PC3"]).round(2)
# 0,25 é só um corte de leitura para a tabela caber; não muda nenhum cálculo.
fortes = cargas[(cargas.abs() >= 0.25).any(axis=1)].copy()
fortes["bloco"] = [bloco_de[c] for c in fortes.index]
# key=abs ordena pelo tamanho da carga, ignorando o sinal.
fortes.sort_values("PC1", key=abs, ascending=False)

In [ ]:
# transform() dá a posição de cada município nas componentes (os "escores").
escores = pca.transform(W)
# A urbanização não é eixo do gráfico, entra como cor: é um teste visual de
# que a PC1 é mesmo o eixo socioeconômico que as cargas sugerem.
fig = plot_pc1_pc2(escores, modelagem["taxa_urbanizacao"],
                   "taxa de urbanização (%)", pca.explained_variance_ratio_)
salvar(fig, "figura_pc1_pc2")

A **PC1 é um eixo de condição socioeconômica**: alfabetização, urbanização,
coleta de lixo, renda e esgoto têm as maiores cargas, todas com o mesmo
sinal. Por isso colorir os pontos pela urbanização mostra um degradê
organizado da esquerda para a direita — que é a confirmação visual da
leitura das cargas. A **PC2** contrasta as notas de saúde e educação do IEGM
(cargas positivas) com as taxas de roubo (cargas negativas).

Mais importante que isso: no gráfico os municípios formam **uma nuvem
contínua**, sem ilhas separadas. Não existem grupos naturais esperando para
serem descobertos. Os clusters da próxima entrega vão ser cortes dentro de
um gradiente, e a silhueta — uma medida de 0 a 1 de quão separados os grupos
ficaram — provavelmente vai dar baixa. Isso não invalida o resultado, mas
muda o que se pode afirmar a partir dele, e é melhor saber disso agora do
que interpretar mal o número depois.

## 6. A matriz que vai para a clusterização

O `matriz_modelagem.csv` tem os 644 municípios com as 22 features já
transformadas, padronizadas e com o peso aplicado, mais as duas colunas de
identificação. É o arquivo que a próxima entrega lê, e é a única saída deste
notebook além das figuras.

In [ ]:
saida = W.copy()
# Identificação na frente das features, para dar para conferir linha a linha
# depois sem precisar cruzar com outro arquivo.
saida.insert(0, "codigo_ibge", modelagem["codigo_ibge"])
saida.insert(1, "municipio", modelagem["municipio"])
saida.to_csv(DATA_PROCESSED / "matriz_modelagem.csv", index=False)
print(f"-> matriz_modelagem.csv: {saida.shape[0]} x {saida.shape[1]}")

In [ ]:
# Resumo das decisões desta fase, num formato que dá para conferir de relance
# e copiar para o artigo.
# cumsum() acumula a variância explicada; o >= 0.80 vira uma lista de
# verdadeiro/falso e o argmax devolve a posição do primeiro verdadeiro. O + 1
# é porque a contagem de posições começa em zero.
k80 = int(np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.80)) + 1
pd.DataFrame({
    "passo": ["subconjunto", "log1p", "padronização", "peso por bloco", "PCA"],
    "decisão": [
        f"{len(modelagem)} municípios (sem a capital)",
        f"{len(log_cols)} colunas: taxas criminais e valores em R$",
        "StandardScaler, ajustado neste subconjunto (ver apêndice 03b)",
        "1/√n: " + ", ".join(f"{b} {peso[[c for c in features if bloco_de[c] == b][0]]:.3f}"
                              for b in ORDEM_BLOCOS),
        f"{k80} componentes para 80% da variância",
    ],
})